# Chapter 28 — Generative Adversarial Networks I: Adversarial Training, DCGAN & Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part4_transformers_genai/ch28_generative_adversarial_networks_i/ch28_notebook.ipynb)

Companion notebook for **Chapter 28** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- A GAN is two networks in competition: the generator makes fakes, the discriminator judges them.
- The generator never sees a real image directly; it improves only through the discriminator's feedback.
- Because the two networks pull in opposite directions, the objective rises and falls during training — that is normal, not a bug.
- Transposed convolution (an FSCL) upsamples, which is how a small noise vector becomes a full-size image.
- Mode collapse destroys diversity; the Inception Score and FID are the standard ways to measure generation quality.

## Example 1: The GAN Minimax Objective

This computes the GAN objective V(D,G) numerically for several scenarios, showing it is high when the discriminator wins and low when the generator wins, with the theoretical equilibrium at 2*log(0.5) = -1.386. The second half walks through one mini-batch training step exactly as the algorithm slide describes, verifying that the discriminator's gradient ascent raises V and the generator's gradient descent lowers it.

In [1]:
# Chapter 28 - Example 1: GAN minimax objective computed numerically
import numpy as np
np.random.seed(0)

print("=== The GAN Minimax Optimization Problem ===\n")
print("min_thetaG  max_thetaD   E_p(x)[log D(x)] + E_p(z)[log(1 - D(G(z)))]\n")
print("  Discriminator D: MAXIMIZE it (gradient ASCENT)")
print("  Generator     G: MINIMIZE it (gradient DESCENT)")
print("  Desired D output: 1 for a real image, 0 for a synthetic image\n")

def objective(D_real, D_fake):
    """V(D,G) for a mini-batch"""
    return np.mean(np.log(D_real + 1e-12)) + np.mean(np.log(1 - D_fake + 1e-12))

M = 6   # mini-batch size
scenarios = {
  "Untrained D (outputs ~0.5 for everything)":
      (np.full(M, 0.50), np.full(M, 0.50)),
  "Strong D (real->0.95, fake->0.05): G is losing":
      (np.full(M, 0.95), np.full(M, 0.05)),
  "Strong G (D fooled: fake->0.90)":
      (np.full(M, 0.60), np.full(M, 0.90)),
  "Equilibrium (D cannot tell: both 0.5)":
      (np.full(M, 0.50), np.full(M, 0.50)),
}

print(f"{'Scenario':<48} {'V(D,G)':>10}  {'Reading'}")
print("-" * 88)
for name, (dr, df) in scenarios.items():
    V = objective(dr, df)
    if "Strong D" in name:   read = "D winning (V high)"
    elif "Strong G" in name: read = "G winning (V low)"
    else:                    read = "V = 2*log(0.5) = -1.386"
    print(f"  {name:<46} {V:>10.4f}  {read}")

print(f"\nTheoretical equilibrium value: 2*log(0.5) = {2*np.log(0.5):.4f}")
print("At equilibrium p(x_synthetic) ~ p(x_real), so D outputs 0.5 everywhere.\n")

# ---- Mini-batch training step (the standard algorithm) ----
print("=== Mini-Batch Mode Training of a GAN (one step) ===\n")
theta_D, theta_G = 0.0, 0.0     # scalar stand-ins for parameter vectors
eta = 0.1
D_real = np.random.uniform(0.55, 0.75, M)
D_fake = np.random.uniform(0.25, 0.45, M)

print("Step 3.1  Sample M random noise vectors z ~ N(0, sigma^2 I)")
z = np.random.randn(M, 4)
print(f"          z shape = {z.shape}  (M={M} noise vectors of dim 4)")

print("\nStep 3.2  Update DISCRIMINATOR by GRADIENT ASCENT")
V_before = objective(D_real, D_fake)
# ascent: D pushes D(x)->1 and D(x~)->0
D_real_new = np.clip(D_real + eta * (1 - D_real), 0, 1)
D_fake_new = np.clip(D_fake - eta * D_fake, 0, 1)
V_after_D = objective(D_real_new, D_fake_new)
print(f"          V before = {V_before:.4f}")
print(f"          V after  = {V_after_D:.4f}   (INCREASED -> ascent worked)")

print("\nStep 3.3  Update GENERATOR by GRADIENT DESCENT")
# descent: G pushes D(x~)->1
D_fake_G = np.clip(D_fake_new + 3 * eta * (1 - D_fake_new), 0, 1)
V_after_G = objective(D_real_new, D_fake_G)
print(f"          V before = {V_after_D:.4f}")
print(f"          V after  = {V_after_G:.4f}   (DECREASED -> descent worked)")

print("\nStep 4    Repeat until convergence.")
print("\nNote the adversarial tension: D pushes V up, G pushes V down.")
print("This is why the objective goes up AND down during GAN training,")
print("unlike ordinary supervised learning where loss decreases steadily.")

=== The GAN Minimax Optimization Problem ===

min_thetaG  max_thetaD   E_p(x)[log D(x)] + E_p(z)[log(1 - D(G(z)))]

  Discriminator D: MAXIMIZE it (gradient ASCENT)
  Generator     G: MINIMIZE it (gradient DESCENT)
  Desired D output: 1 for a real image, 0 for a synthetic image

Scenario                                             V(D,G)  Reading
----------------------------------------------------------------------------------------
  Untrained D (outputs ~0.5 for everything)         -1.3863  V = 2*log(0.5) = -1.386
  Strong D (real->0.95, fake->0.05): G is losing    -0.1026  D winning (V high)
  Strong G (D fooled: fake->0.90)                   -2.8134  G winning (V low)
  Equilibrium (D cannot tell: both 0.5)             -1.3863  V = 2*log(0.5) = -1.386

Theoretical equilibrium value: 2*log(0.5) = -1.3863
At equilibrium p(x_synthetic) ~ p(x_real), so D outputs 0.5 everywhere.

=== Mini-Batch Mode Training of a GAN (one step) ===

Step 3.1  Sample M random noise vectors z ~ N(0, sigm

## Example 2: Transposed Convolution (FSCL) in the DCGAN Generator

This shows why the generator needs a different operation from the discriminator. Ordinary convolution shrinks the 4x4 input to a 3x3 feature map - and you can check each value by hand. We then build the convolution matrix C, check e = Cx, and use C-transpose to upsample back to 4x4. The result confirms that the shape is restored but the values are not the original ones.

In [2]:
# Chapter 28 - Example 2: Transposed Convolution (FSCL) used in the DCGAN generator
import numpy as np
np.set_printoptions(suppress=True)

print("=== Transposed Convolution (Up-Convolution / FSCL) ===\n")
print("The DISCRIMINATOR uses ordinary convolution: image -> smaller feature map.")
print("The GENERATOR must go the other way: small input -> larger image.")
print("It uses Transposed Convolution, also called Fractional-Strided")
print("Convolution (stride S < 1), which is why the layer is named FSCL.\n")

# ---------- Ordinary (down) convolution using the convolution matrix ----------
x = np.array([[5, 9, 3, 1],
              [7, 2, 6, 4],
              [1, 8, 5, 2],
              [3, 0, 7, 9]], dtype=float)  # 4x4 input
K = np.array([[0.2, -0.5],
              [0.7, -0.3]], dtype=float)            # 2x2 kernel

def conv2d(img, ker, stride=1):
    kh, kw = ker.shape
    oh = (img.shape[0]-kh)//stride + 1
    ow = (img.shape[1]-kw)//stride + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = np.sum(img[i*stride:i*stride+kh, j*stride:j*stride+kw] * ker)
    return out

feat = conv2d(x, K, 1)
print("Input image x (4x4):"); print(x.astype(int))
print("\nKernel K (2x2):"); print(K)
print(f"\nDown-convolution output (3x3)  W2 = (W1-F)/S + 1 = (4-2)/1+1 = 3:")
print(np.round(feat, 2))
print("  -> W2 < W1 : the feature map SHRANK\n")

# ---------- Build the convolution matrix C, then use C^T to upsample ----------
def convolution_matrix(ker, in_h, in_w):
    kh, kw = ker.shape
    oh, ow = in_h-kh+1, in_w-kw+1
    C = np.zeros((oh*ow, in_h*in_w))
    for i in range(oh):
        for j in range(ow):
            row = i*ow + j
            for a in range(kh):
                for b in range(kw):
                    C[row, (i+a)*in_w + (j+b)] = ker[a, b]
    return C

C = convolution_matrix(K, 4, 4)
x_flat = x.reshape(-1)
e_flat = C @ x_flat
print(f"Convolution matrix C shape: {C.shape}  (9 outputs x 16 inputs)")
print(f"Verify  e_bar = C x_bar  matches the direct convolution: "
      f"{np.allclose(e_flat.reshape(3,3), feat)}\n")

# ---------- Transposed convolution: x_tilde = C^T e ----------
print("=== Transposed Convolution: x_tilde_bar = C^T e_bar ===\n")
up_flat = C.T @ e_flat
up = up_flat.reshape(4, 4)
print(f"C^T shape: {C.T.shape}  (16 outputs x 9 inputs) -> UPSAMPLES 3x3 back to 4x4")
print("Result (4x4):")
print(np.round(up, 2))
print(f"\nIs the result equal to the original x? {np.allclose(up, x)}")
print("NO - transposed convolution restores the SHAPE, not the VALUES.")
print("(Note: x_tilde is not the same as x.)\n")

# ---------- Generator in DCGAN ----------
print("=== Generator in DCGAN ===\n")
print("A 100-dim noise vector z is progressively upsampled by FSCL layers:")
stages = [("Unflattened input", "4x4x1024"), ("FSCL 1", "8x8x512"),
          ("FSCL 2", "16x16x256"), ("FSCL 3", "32x32x128"),
          ("FSCL 4 / Conv", "64x64x3  (generated image)")]
for name, shape in stages:
    print(f"  {name:<20} -> {shape}")
print("\nEach FSCL doubles height and width while reducing depth,")
print("turning random noise z into a full-size synthetic image G(z).")

=== Transposed Convolution (Up-Convolution / FSCL) ===

The DISCRIMINATOR uses ordinary convolution: image -> smaller feature map.
The GENERATOR must go the other way: small input -> larger image.
It uses Transposed Convolution, also called Fractional-Strided
Convolution (stride S < 1), which is why the layer is named FSCL.

Input image x (4x4):
[[5 9 3 1]
 [7 2 6 4]
 [1 8 5 2]
 [3 0 7 9]]

Kernel K (2x2):
[[ 0.2 -0.5]
 [ 0.7 -0.3]]

Down-convolution output (3x3)  W2 = (W1-F)/S + 1 = (4-2)/1+1 = 3:
[[ 0.8 -0.1  3.1]
 [-1.3  1.5  2.1]
 [-1.7 -3.   2.2]]
  -> W2 < W1 : the feature map SHRANK

Convolution matrix C shape: (9, 16)  (9 outputs x 16 inputs)
Verify  e_bar = C x_bar  matches the direct convolution: True

=== Transposed Convolution: x_tilde_bar = C^T e_bar ===

C^T shape: (16, 9)  (16 outputs x 9 inputs) -> UPSAMPLES 3x3 back to 4x4
Result (4x4):
[[ 0.16 -0.42  0.67 -1.55]
 [ 0.3   0.64  1.87 -1.98]
 [-1.25  1.69  2.96 -1.73]
 [-1.19 -1.59  2.44 -0.66]]

Is the result equal to t

## Example 3: Mode Collapse and the Inception Score

The first half shows mode collapse: a healthy GAN covers all ten digit modes while a collapsed one produces only two. The second half builds the Inception Score using KL divergence and shows that a high score requires BOTH confident per-image predictions AND a diverse marginal distribution - mode collapse destroys the second, blurry generation destroys the first.

In [3]:
# Chapter 28 - Example 3: Mode collapse + Inception Score (GAN evaluation)
import numpy as np
np.random.seed(5)

print("=== Evaluating a GAN: Creativity, Inheritance, Diversity ===\n")
print("A GAN image generator is judged on three aspects:")
print("  Creativity  : generated images are NOT duplicates of real images")
print("  Inheritance : generated images retain key features of real images")
print("  Diversity   : generated images differ from one another\n")

# ---------- Mode collapse ----------
print("=== Mode Collapse (Mode Diversity Problem) ===\n")
print("The generator maps ALL noise vectors z to a small subset of outputs.")
print("Example: a GAN trained on handwritten digits generates only a few digits.\n")

n = 1000
healthy  = np.random.choice(10, n)                       # all 10 digits
collapsed = np.random.choice([3, 8], n, p=[0.6, 0.4])    # only 2 digits

def digit_hist(labels, title):
    counts = np.bincount(labels, minlength=10)
    print(f"  {title}")
    for d in range(10):
        bar = "#" * int(counts[d] / 8)
        print(f"    digit {d}: {counts[d]:>4}  {bar}")
    modes = int((counts > 0).sum())
    print(f"    distinct modes generated: {modes}/10\n")
    return counts

c_ok  = digit_hist(healthy,   "Healthy GAN (good mode coverage):")
c_bad = digit_hist(collapsed, "Mode-collapsed GAN:")

print("The fix: the Least-Squares GAN (LSGAN) replaces the")
print("cross-entropy discriminator loss with sum-of-squared errors and is")
print("shown to avoid mode collapse. The Wasserstein GAN instead changes")
print("the criterion to the Wasserstein (Earth Mover's) distance, which")
print("measures how far apart p(x~) and p(x) are.\n")

# ---------- Inception Score ----------
print("=== Inception Score (IS) via KL Divergence ===\n")
print("Generated images are fed to an Inception network. For each image we get")
print("p(y|x) over class labels. IS = exp( E_x [ KL( p(y|x) || p(y) ) ] )\n")

def inception_score(P):
    """P: (n_images x n_classes) matrix of class probabilities."""
    p_y = P.mean(axis=0)                                  # marginal p(y)
    kl  = np.sum(P * (np.log(P + 1e-12) - np.log(p_y + 1e-12)), axis=1)
    return float(np.exp(kl.mean())), p_y

K = 5
# Case 1: sharp per-image predictions AND diverse classes -> high IS
sharp_diverse = np.full((100, K), 0.02)
for i in range(100):
    sharp_diverse[i, i % K] = 0.92
# Case 2: sharp but ALL the same class (mode collapse) -> low IS
sharp_collapsed = np.full((100, K), 0.02)
sharp_collapsed[:, 0] = 0.92
# Case 3: blurry/uncertain predictions -> low IS
blurry = np.full((100, K), 1.0 / K)

for name, P in [("Sharp + diverse (good GAN)", sharp_diverse),
                ("Sharp but mode-collapsed",   sharp_collapsed),
                ("Blurry / unrecognisable",    blurry)]:
    P = P / P.sum(axis=1, keepdims=True)
    score, p_y = inception_score(P)
    print(f"  {name:<30} IS = {score:.4f}   marginal p(y) = {np.round(p_y,3)}")

print(f"\n  Maximum possible IS for {K} classes = {K}")
print("\nInterpretation:")
print("  High IS needs BOTH  (a) confident per-image class predictions")
print("                 and  (b) a uniform marginal over classes (diversity).")
print("  Mode collapse kills (b), blurry generation kills (a).")
print("\nThe other standard metric is the Frechet Inception Distance (FID),")
print("which compares the feature distributions of real and generated images.")

=== Evaluating a GAN: Creativity, Inheritance, Diversity ===

A GAN image generator is judged on three aspects:
  Creativity  : generated images are NOT duplicates of real images
  Inheritance : generated images retain key features of real images
  Diversity   : generated images differ from one another

=== Mode Collapse (Mode Diversity Problem) ===

The generator maps ALL noise vectors z to a small subset of outputs.
Example: a GAN trained on handwritten digits generates only a few digits.

  Healthy GAN (good mode coverage):
    digit 0:  112  ##############
    digit 1:  119  ##############
    digit 2:   92  ###########
    digit 3:   98  ############
    digit 4:   95  ###########
    digit 5:   96  ############
    digit 6:  102  ############
    digit 7:   92  ###########
    digit 8:   91  ###########
    digit 9:  103  ############
    distinct modes generated: 10/10

  Mode-collapsed GAN:
    digit 0:    0  
    digit 1:    0  
    digit 2:    0  
    digit 3:  592  #########